# Module 5 — Inverse Kinematics
## Unit 4 — From Geometry to Numerical IK
### Lesson 4.1 — When Closed Form Runs Out

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
A redundant planar 3-link arm: a whole family of solutions for one target.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

def family(x,y,L1,L2,L3,theta3_values):
    out=[]
    for th3 in theta3_values:
        # wrist (joint-3 base) sits back from target along the final link direction th3
        wx,wy = x - L3*np.cos(th3), y - L3*np.sin(th3)
        r2=wx*wx+wy*wy
        if (L1-L2)**2-1e-9 <= r2 <= (L1+L2)**2+1e-9:
            c2=(r2-L1*L1-L2*L2)/(2*L1*L2); c2=max(-1,min(1,c2)); t2=np.arccos(c2)
            t1=np.arctan2(wy,wx)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
            out.append((t1,t2,th3))
    return out

L1=L2=L3=0.3
fam=family(0.6,0.0,L1,L2,L3,np.radians([0,20,-20,40]))
print("found",len(fam),"configurations reaching (0.6, 0):")
for (t1,t2,t3) in fam:
    print("  ",np.round(np.degrees([t1,t2,t3]),1))


## Coding Exercise (§8)
Exhibit the solution family; each whole-arm pose reaches the target.


In [ ]:
# YOUR CODE HERE
def fk3(t1,t2,t3,L1,L2,L3):
    x=L1*np.cos(t1)+L2*np.cos(t1+t2)+L3*np.cos(t1+t2+t3)
    y=L1*np.sin(t1)+L2*np.sin(t1+t2)+L3*np.sin(t1+t2+t3)
    return np.array([x,y])
# NOTE: th3 in family is the absolute final-link angle = t1+t2+t3_rel
for (t1,t2,th3) in fam:
    t3_rel=th3-(t1+t2)
    assert np.allclose(fk3(t1,t2,t3_rel,L1,L2,L3),[0.6,0.0],atol=1e-9)
assert len(fam)>=3   # a continuum, sampled
print("redundant family OK")


## Check your work


In [ ]:
assert len(family(0.6,0.0,L1,L2,L3,np.radians([0,20,-20,40])))>=3
print("All checks passed.")
